# MINT-TTS — the homograph experiment

**Minimal Inference Needed for Text-to-Speech.** Does a speech model learn to spend
*more computation* on words whose pronunciation is genuinely ambiguous?

This notebook runs the configuration in which that question is actually testable:

| | why |
|---|---|
| **character input** | with IPA, espeak already picks a pronunciation, so nothing is left to disambiguate. The previous run measured exactly that: one encoder step was as good as eight. |
| **HiFi-GAN vocoder** | Griffin-Lim makes everything sound robotic, so you cannot judge the model by ear |
| **checkpoints on Drive** | a Colab disconnect must not cost 30k steps |
| **homograph probe** | measures whether the extra compute *changes the pronunciation*, not just where it went |

**Runtime → Change runtime type → GPU** before running anything.

## 1. Install

In [ ]:
REPO = 'https://github.com/MohammedAly22/MINT-TTS.git'

import os, pathlib
if not pathlib.Path('MINT-TTS').exists():
    !git clone -q $REPO MINT-TTS
os.chdir('MINT-TTS')
!git pull -q
!pip install -q -r requirements.txt

import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

## 2. Checkpoints on Google Drive

Colab reclaims the VM without warning and everything under `/content` goes with it.
Everything is written to `Mint-TTS/experiments` in your Drive instead, so a run that
dies at step 30k resumes from step 28k.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

RUN_DIR = '/content/drive/MyDrive/Mint-TTS/experiments'
!mkdir -p "$RUN_DIR"
os.environ['RUN_DIR'] = RUN_DIR
print('checkpoints and TensorBoard logs ->', RUN_DIR)

## 3. HiFi-GAN

The default vocoder is Griffin-Lim, which is why generated audio sounds phasey and
metallic even when the model is fine. HiFi-GAN is a trained neural vocoder — same
mel settings (22.05 kHz, hop 256, 80 mels), V1 architecture.

It is **frozen and shared by every experiment**, so any quality difference you hear
between runs comes from the acoustic model, never from the vocoder.

In [ ]:
!pip install -q huggingface_hub
!python scripts/download_vocoder.py \
    --hf-repo speechbrain/tts-hifigan-ljspeech --hf-file generator.ckpt

# 'OK: loaded ...' above means real weights were applied, not a random init.

## 4. Which frontend, and why it decides the experiment

In [ ]:
!python scripts/inspect_frontend.py

# 'same' = the frontend committed to ONE pronunciation for both contexts.
# espeak does this for most pairs, which is why we train on characters:
# the model then has to resolve the ambiguity itself, and only then can
# spending compute on it possibly help.

## 5. LJSpeech (2.6 GB, ~3 min)

In [ ]:
import pathlib
if not pathlib.Path('data/LJSpeech-1.1/metadata.csv').exists():
    !mkdir -p data
    !curl -sL -o /tmp/ljs.tar.bz2 https://data.keithito.com/data/speech/LJSpeech-1.1.tar.bz2
    !tar -xjf /tmp/ljs.tar.bz2 -C data/
!python scripts/prepare_dataset.py --dataset ljspeech --root data/LJSpeech-1.1

## 6. Preprocess (~2–3 min)

Character input needs its own feature directory: the vocabulary differs from the
IPA one and the two are not interchangeable.

In [ ]:
!python scripts/preprocess.py --config configs/experiment_char.yaml --workers 4

import json
stats = json.load(open('data/preprocessed/ljspeech_char/stats.json'))
print(f"{stats['n_utterances']} utterances, {stats['total_hours']:.1f} h, "
      f"vocab {stats['vocab_size']} ({stats['input_type']})")

## 7. TensorBoard in its own browser tab

Run this, then click the link. A separate tab is far easier to live in than the
embedded widget — you will be switching between SCALARS, IMAGES and AUDIO a lot.

In [ ]:
import subprocess, time
subprocess.Popen(['tensorboard', '--logdir', RUN_DIR, '--port', '6006',
                  '--bind_all', '--reload_multifile', 'true'])
time.sleep(6)
from google.colab import output
output.serve_kernel_port_as_window(6006)   # <- click the link that appears

## 8. Train

Routing is held at **full depth** until `loss.compute.warmup_steps`, so the model
learns to speak before any compute pressure arrives. Until then `enc_depth` sits at
8 and `compute/lambda` is 0 — that is intended, not a stall.

**Watch, in this order:**

1. `align/entropy_ratio` must fall from ~0.65 to below 0.4. If it hovers near 1.0 the
   aligner is at chance and nothing downstream means anything — stop.
2. `val/mcd_vs_chance` must drop well below 1.0 (~0.3 is good). ≥1.0 means the audio
   carries no information about *which* sentence was asked for.
3. After warmup: `compute/encoder_depth_spread`. **This is the experiment.** Near 0 means
   the router picked a constant — no allocation. Clearly above 0 means it is differentiating.
4. `homograph/divergence_ratio` > 1 means ambiguous words are rendered *differently*
   across contexts, beyond ordinary variation.
5. `probe/contrast` > 0 and `probe/length_corr` well below 1.

In **AUDIO** you get every probe sentence, plus `homograph/<word>/audio_a` and
`audio_b` side by side — listen to whether "record" actually changes.

In [ ]:
!python scripts/train.py --config configs/experiment_char.yaml \
    --override train.output_dir="$RUN_DIR" train.save_every=2000 \
               train.batch_size=16 train.amp=true train.max_steps=200000 \
               loss.compute.warmup_steps=8000 loss.compute.ramp_steps=4000 \
               log.probe_every=2000 log.figure_every=2000 log.log_audio=true

In [ ]:
# Resume after a disconnect
# !python scripts/train.py --config configs/experiment_char.yaml \
#     --override train.output_dir="$RUN_DIR" \
#     --resume "$RUN_DIR/experiment_char/checkpoints/best.pt"

## 9. Is there anything to allocate?

Run this on a mid-training checkpoint. It measures quality as a function of *forced*
depth, per utterance, and derives C\* — the cheapest depth that still reaches 98% of
peak quality.

**`c_star_encoder_unique` is the number that matters.** If it is 1, every utterance
has the same requirement, there is no headroom, and no router can invent any. That is
exactly what killed the previous run — check it early rather than after 30k steps.

In [ ]:
!python scripts/compute_curve.py \
    --checkpoint "$RUN_DIR/experiment_char/checkpoints/best.pt" \
    --index data/preprocessed/ljspeech_char/test.jsonl --limit 100

## 10. Listen, and see where the compute went

In [ ]:
from IPython.display import Audio, display
from mint_tts.inference.synthesize import Synthesizer
from mint_tts.utils import plotting

syn = Synthesizer.from_checkpoint(f'{RUN_DIR}/experiment_char/checkpoints/best.pt')

PAIRS = [('read',   'I read a book yesterday.',       'I will read a book tomorrow.'),
         ('record', 'Please record the album today.', 'She bought the record today.'),
         ('lead',   'He will lead the team.',         'The pipe is made of lead.')]

for word, a, b in PAIRS:
    print(f'=== {word} ===')
    for text in (a, b):
        res = syn(text, quality=0.9)
        print(' ', text)
        if res.wav is not None:
            display(Audio(res.wav.cpu().numpy(), rate=res.sample_rate))
        plotting.show(plotting.plot_word_complexity(
            res.encoded.words, res.word_complexity, title=text[:60],
            highlight=[word]))

In [ ]:
# Tongue twisters and normalisation, for listening
for text in ['She sells seashells by the seashore.',
             'Peter Piper picked a peck of pickled peppers.',
             'Six sticky skeletons stacked six thick bricks.',
             'Doctor Smith paid $1,250.75 on March 3rd, 1987.',
             'Call +1 (555) 123-4567 or email a.smith@mit.edu before 9:05 a.m.']:
    res = syn(text, quality=0.9)
    print(res.summary())
    if res.wav is not None:
        display(Audio(res.wav.cpu().numpy(), rate=res.sample_rate))
    plotting.show(plotting.plot_token_complexity(
        res.encoded.tokens, res.token_complexity, title=text[:60],
        max_steps=syn.model.encoder.max_steps,
        words=res.encoded.words, word_ids=res.encoded.word_ids))

## 11. The budget knob: C*(x, q)

Only varies once routing has been trained, i.e. well past `warmup_steps`.

In [ ]:
text = 'The record is broken by the record broker.'
qs, cs = [], []
for q in [0.1, 0.3, 0.5, 0.7, 0.9, 1.0]:
    r = syn(text, quality=q)
    qs.append(q); cs.append(float(r.token_complexity.mean()))
    print(f'q={q:<4} compute={cs[-1]:.3f}  FLOPs={r.flops.total:.2e}  '
          f'saving={r.flops.saving*100:5.1f}%  {r.latency_ms:6.1f} ms')
plotting.show(plotting.plot_compute_curve(
    cs, qs, title='requested quality vs allocated compute'))

## 12. If the encoder turns out to have no headroom

If `c_star_encoder_unique` is 1 again, the encoder is over-provisioned and the place
to look is the **decoder** — it is 71–78% of the FLOPs and is fixed in
`experiment_char`. `experiment_char_unified` makes it adaptive too, raising the
ceiling on total saving from ~15% to ~74%:

```bash
python scripts/train.py --config configs/experiment_char_unified.yaml \
    --override train.output_dir="$RUN_DIR"
```

New to the terminology? [`docs/HOW_IT_WORKS.md`](../docs/HOW_IT_WORKS.md) explains every
term used here — probe, C\*, MCD, mcd/chance, ponder, and the optimisation being solved.